## 1. Imports and API setup

In [ ]:
from pathlib import Path

from video_retrieval import (
    RetrievalResources,
    VideoRetrievalPipeline,
    get_openrouter_api_key,
    split_video_hierarchically,
)

from video_retrieval.embeddings import (
    embed_all_scales,
    load_multiscale_video_index,
)

from video_retrieval.metadata import (
    build_metadata_faiss,
    embed_metadata_records,
    generate_metadata,
    load_jsonl,
    load_metadata_index,
)

from video_retrieval.transcript import (
    build_bm25,
    build_transcript_faiss,
    create_transcript_windows,
    embed_transcript_windows,
    load_transcript_index,
    transcribe_video,
)

get_openrouter_api_key(prompt_if_missing=True)

## 2. Project configuration

In [ ]:
VID_NAME = "video_16"
VIDEO_PATH = Path("/Users/eddie/Downloads/" + VID_NAME+".mp4")

CHUNK_DIR = Path(VID_NAME+"_chunks")
VISUAL_INDEX_DIR = Path(VID_NAME+"_embedding_indices")

METADATA_SCALE = "medium"
METADATA_JSONL = Path(VID_NAME+"_metadata/medium_metadata.jsonl")
METADATA_INDEX_DIR = Path(VID_NAME+"_metadata_index")

TRANSCRIPT_DIR = Path(VID_NAME+"_transcript")
TRANSCRIPT_INDEX_DIR = Path(VID_NAME+"_transcript_index")
FINAL_RESULTS_DIR = Path(VID_NAME+"_final_results")

# Set these to True only when you need to rebuild or build a video for the first time. (Set to False after first run)
REBUILD_VISUAL_INDEX = False
REBUILD_METADATA = False
REBUILD_TRANSCRIPT = False

# Hierarchical visual-search settings.
VISUAL_BEAM_MULTIPLIER = 4
VISUAL_MINIMUM_BEAM = 24
VISUAL_CONTEXT_WEIGHT = 0.30
VISUAL_PARENT_PADDING = 1.0

## 3. Build the hierarchical video manifest

In [4]:
manifest = split_video_hierarchically(
    video_path=VIDEO_PATH,
    output_dir=CHUNK_DIR,
    export_clips=False,
)

print("Video duration:", manifest["video"]["duration"])
print("Available chunk scales:", list(manifest["chunks"].keys()))

for scale, chunks in manifest["chunks"].items():
    if chunks:
        durations = [
            float(chunk.get("duration", chunk["end"] - chunk["start"]))
            for chunk in chunks
        ]
        print(
            f"{scale:>12}:",
            len(chunks),
            "chunks |",
            f"median duration={sorted(durations)[len(durations)//2]:.2f}s",
        )

Video: video_16.mp4
Duration: 1018.01 sec
Duration: 16.97 min

coarse: 16 chunks
medium: 67 chunks
fine: 254 chunks

Manifest saved to: video_16_chunks\manifest.json
Video duration: 1018.011328
Available chunk scales: ['coarse', 'medium', 'fine']
      coarse: 16 chunks | median duration=120.00s
      medium: 67 chunks | median duration=30.00s
        fine: 254 chunks | median duration=8.00s


## 4. Build or load the multi-scale visual index

In [5]:
if REBUILD_VISUAL_INDEX:
    video_index, video_metadata = embed_all_scales(
        manifest=manifest,
        save_dir=VISUAL_INDEX_DIR,
        beam_multiplier=VISUAL_BEAM_MULTIPLIER,
        minimum_beam=VISUAL_MINIMUM_BEAM,
        context_weight=VISUAL_CONTEXT_WEIGHT,
        parent_padding=VISUAL_PARENT_PADDING,
    )
else:
    video_index, video_metadata = load_multiscale_video_index(
        save_dir=VISUAL_INDEX_DIR,
        beam_multiplier=VISUAL_BEAM_MULTIPLIER,
        minimum_beam=VISUAL_MINIMUM_BEAM,
        context_weight=VISUAL_CONTEXT_WEIGHT,
        parent_padding=VISUAL_PARENT_PADDING,
    )

print("Hierarchical visual index ready.")
print("Scale order (coarse → fine):", video_index.scale_order)
print("Output scale:", video_index.output_scale)
print("Finest-scale searchable chunks:", video_index.ntotal)

Hierarchical visual index ready.
Scale order (coarse → fine): ['coarse', 'medium', 'fine']
Output scale: fine
Finest-scale searchable chunks: 254


### Optional: inspect the inferred hierarchy

In [6]:
for scale in video_index.scale_order:
    records = video_index.metadata_by_scale[scale]
    print(f"\n{scale}: {len(records)} chunks")

    for record in records[:3]:
        print(
            {
                "chunk_id": record.get("chunk_id"),
                "start": record.get("start"),
                "end": record.get("end"),
                "parent_id": record.get("parent_id"),
                "parent_ids": record.get("parent_ids", []),
                "num_children": len(record.get("child_ids", [])),
            }
        )


coarse: 16 chunks
{'chunk_id': 'coarse_000000', 'start': 0.0, 'end': 120.0, 'parent_id': None, 'parent_ids': [], 'num_children': 8}
{'chunk_id': 'coarse_000001', 'start': 60.0, 'end': 180.0, 'parent_id': None, 'parent_ids': [], 'num_children': 9}
{'chunk_id': 'coarse_000002', 'start': 120.0, 'end': 240.0, 'parent_id': None, 'parent_ids': [], 'num_children': 9}

medium: 67 chunks
{'chunk_id': 'medium_000000', 'start': 0.0, 'end': 30.0, 'parent_id': 'coarse_000000', 'parent_ids': ['coarse_000000'], 'num_children': 8}
{'chunk_id': 'medium_000001', 'start': 15.0, 'end': 45.0, 'parent_id': 'coarse_000000', 'parent_ids': ['coarse_000000'], 'num_children': 10}
{'chunk_id': 'medium_000002', 'start': 30.0, 'end': 60.0, 'parent_id': 'coarse_000000', 'parent_ids': ['coarse_000000'], 'num_children': 9}

fine: 254 chunks
{'chunk_id': 'fine_000000', 'start': 0.0, 'end': 8.0, 'parent_id': 'medium_000000', 'parent_ids': ['medium_000000'], 'num_children': 0}
{'chunk_id': 'fine_000001', 'start': 4.0, '

## 5. Build or load searchable VLM metadata

In [7]:
if REBUILD_METADATA:
    generate_metadata(
        manifest,
        scale=METADATA_SCALE,
        output_path=METADATA_JSONL,
    )

    metadata_records = load_jsonl(METADATA_JSONL)

    metadata_embeddings, metadata_index_records = embed_metadata_records(
        metadata_records,
        save_dir=METADATA_INDEX_DIR,
    )

    metadata_index = build_metadata_faiss(
        metadata_embeddings,
        save_path=METADATA_INDEX_DIR / "medium_metadata.faiss",
    )
else:
    metadata_index, metadata_index_records = load_metadata_index(
        save_dir=METADATA_INDEX_DIR,
    )

print("Metadata index ready:", len(metadata_index_records), "records")

Metadata index ready: 67 records


## 6. Build or load transcript semantic + BM25 indexes

In [8]:
if REBUILD_TRANSCRIPT:
    words, segments = transcribe_video(
        manifest,
        output_dir=TRANSCRIPT_DIR,
        language="en",
    )

    transcript_windows = create_transcript_windows(
        words,
        manifest["video"]["duration"],
    )

    transcript_windows = [
        window
        for window in transcript_windows
        if window["text"].strip()
    ]

    transcript_embeddings, transcript_metadata = embed_transcript_windows(
        transcript_windows,
        save_dir=TRANSCRIPT_INDEX_DIR,
    )

    transcript_index = build_transcript_faiss(
        transcript_embeddings,
        save_path=TRANSCRIPT_INDEX_DIR / "transcript.faiss",
    )

    transcript_bm25 = build_bm25(transcript_metadata)
else:
    (
        transcript_index,
        transcript_bm25,
        transcript_metadata,
    ) = load_transcript_index(
        save_dir=TRANSCRIPT_INDEX_DIR,
    )

print("Transcript index ready:", len(transcript_metadata), "windows")

Transcript index ready: 67 windows


## 7. Construct the retrieval pipeline

In [9]:
resources = RetrievalResources(
    manifest=manifest,

    video_index=video_index,
    video_metadata=video_metadata,

    metadata_index=metadata_index,
    metadata_records=metadata_index_records,

    transcript_index=transcript_index,
    transcript_bm25=transcript_bm25,
    transcript_metadata=transcript_metadata,
)

pipeline = VideoRetrievalPipeline(resources)

print("Pipeline ready.")

Pipeline ready.


## 8. Run a query

In [ ]:
PROMPT = "Find all footage of the woman in the black and green jacket."

results = pipeline.retrieve(PROMPT)

print("executor:", results["plan"]["executor"])
print("routing reason:", results["plan"].get("routing_reason", ""))
print("matches:", results["num_matches"])

## 9. Inspect results

In [ ]:
if results["plan"]["executor"] == "visual_text_extraction":
    print("Target specification:")

    for key, value in results["target_spec"].items():
        print(f"  {key}: {value}")

    print(
        "\nUnique readable texts:",
        results["num_unique_readable_texts"],
    )

    for entity in results["text_entities"]:
        print(
            "\nTEXT:",
            entity["text"],
            "confidence=",
            round(entity["confidence"], 3),
        )

        for appearance in entity["appearances"]:
            print(
                "  ",
                appearance["start_timestamp"],
                "→",
                appearance["end_timestamp"],
                "crop=",
                appearance["region_crop_path"],
            )

else:
    print(
        "Target event:",
        results["plan"].get("target_event", PROMPT),
    )

    print("\nEvidence predicates:")
    for predicate in results["plan"].get("evidence_predicates", []):
        print(
            "  -",
            predicate["description"],
            "| role=",
            predicate["role"],
            "| modalities=",
            predicate["modalities"],
            "| required=",
            predicate["required"],
        )

    diagnostics = results.get("diagnostics", {})

    if diagnostics:
        print("\nCandidate-search diagnostics:")
        print(
            "  ",
            diagnostics.get("num_initial_candidates", 0),
            "initial →",
            diagnostics.get("num_candidates", 0),
            "recursive candidates",
        )
        print(
            "  evidence peak score:",
            round(
                diagnostics.get("evidence_peak_score", 0.0),
                3,
            ),
        )

    print("\nFinal matches:")

    for match in results["matches"]:
        print(
            match["match_id"],
            match["start_timestamp"],
            "→",
            match["end_timestamp"],
            "confidence=",
            round(match["confidence"], 3),
            "frames=",
            len(match["frames"]),
            "clip=",
            match["clip_path"],
        )

## 10. Debug the multi-scale visual channel directly

In [ ]:
from video_retrieval.embeddings import search_video

PROMPT = "Find all interactions where an officer reads Miranda rights"

visual_debug = search_video(
    PROMPT,
    index=video_index,
    metadata=video_metadata,
    top_k=100,
)

for result in visual_debug:
    print(
        result["chunk_id"],
        f'{result["start"]:.2f}s → {result["end"]:.2f}s',
        "score=",
        round(result["score"], 4),
        "mode=",
        result.get("retrieval_mode"),
        "scales=",
        result.get("retrieval_scale_order"),
    )

In [ ]:
from video_retrieval.metadata import search_metadata

PROMPT = "Find every moment where someone is exiting a vehicle"

metadata_debug = search_metadata(
    PROMPT,
    index=metadata_index,
    records=metadata_records,
    top_k=50,
)

for result in metadata_debug:
    print(
        result.get("chunk_id"),
        f'{result["start"]:.2f}s → {result["end"]:.2f}s',
        "score=",
        round(result["score"], 4),
        "text=",
        result.get("text"),
    )